# Tiebreak-Level Panel

Reshapes the team-level Grand Slam panel into a **tiebreak-level** panel where each
observation is one team in one specific tiebreak.

| Match type | Team obs | Tiebreak obs |
|---|---|---|
| 1 regular TB (any set 1-5) | 2 | 2 |
| 2 regular TBs | 2 | 4 |
| Match TB (10-pt / advantage-set decider, set 3, 4, or 5) | 2 | 2 |

Sets 4-5 only occur in the 103 best-of-5 Wimbledon matches (2018/2019/2021/2022).
The match tiebreak / advantage-set-decider category can occur in whichever set is
actually the deciding set (3 for best-of-3, 3/4/5 for best-of-5).

Outcome (`won_tb`): did this team win this specific tiebreak?
`tb_type`: '7pt' for a standard 6-6 tiebreak, '10pt' for the match-tiebreak/advantage-set
decider category (see `homophily.ipynb` section 2.2 for why '10pt' here also covers the
no-breaker advantage-set finishes at Wimbledon/Olympics -- kept for continuity with the
original variable name).

Output: `data/atp/tiebreak_panel.csv`


In [1]:
import os
import pandas as pd
import numpy as np

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PANEL_PATH = os.path.join(ROOT, 'data', 'atp', 'team_gs_panel.csv')
OUT_PATH   = os.path.join(ROOT, 'data', 'atp', 'tiebreak_panel.csv')
print('Input: ', PANEL_PATH)
print('Output:', OUT_PATH)

Input:  C:\Users\ALESSANDRO\Documents\GitHub\tennis-homophily\data\atp\team_gs_panel.csv
Output: C:\Users\ALESSANDRO\Documents\GitHub\tennis-homophily\data\atp\tiebreak_panel.csv


## 1. Load team panel — same filters as main analysis

In [2]:
df = pd.read_csv(PANEL_PATH)
print(f'Loaded: {len(df)} team-obs ({len(df)//2} matches)')

# Exclude Olympics (same filter as Tables 3-5)
df = df[df['olympics_tourn'] == 0].copy()
print(f'After excl. Olympics: {len(df)} team-obs ({len(df)//2} matches)')

# Drop rows missing GS experience (same filter as Tables 3-5)
df = df[df['exp_mean'].notna()].copy()
print(f'After exp_mean filter: {len(df)} team-obs')

# Keep only complete match pairs - both team rows must survive all filters.
# Orphaned rows arise when only one team's exp_mean is available; they are fine
# in the match-win regression but break the tiebreak panel's 2-rows-per-TB structure.
pair_size = df.groupby('match_id')['match_id'].transform('count')
df = df[pair_size == 2].copy()
print(f'After requiring complete pairs: {len(df)} team-obs ({len(df)//2} matches)')

# Generate exp_mean_sq if not already present
if 'exp_mean_sq' not in df.columns:
    df['exp_mean_sq'] = df['exp_mean'] ** 2

Loaded: 3752 team-obs (1876 matches)
After excl. Olympics: 3752 team-obs (1876 matches)
After exp_mean filter: 3752 team-obs
After requiring complete pairs: 3752 team-obs (1876 matches)


## 2. Derive per-set tiebreak flags

The team panel has `won_tb_s1` and `won_tb_s2` per team-row. For each match, exactly one
team has `won_tb_s1 = 1` if there was a set-1 tiebreak (the other has 0); if there was no
set-1 tiebreak both teams have 0.  We recover the per-match flag via the max per match.

In [3]:
# Per-match flags: was there a tiebreak in each specific set?
df['has_tb_s1'] = df.groupby('match_id')['won_tb_s1'].transform('max').gt(0).astype(int)
df['has_tb_s2'] = df.groupby('match_id')['won_tb_s2'].transform('max').gt(0).astype(int)
df['has_tb_s3'] = df.groupby('match_id')['won_tb_s3'].transform('max').gt(0).astype(int)
df['has_tb_s4'] = df.groupby('match_id')['won_tb_s4'].transform('max').gt(0).astype(int)
df['has_tb_s5'] = df.groupby('match_id')['won_tb_s5'].transform('max').gt(0).astype(int)
# match_tb is already a match-level flag in the panel: True for whichever set was
# actually the deciding set, if it ended in a 10-pt/advantage-set finish. match_tb_set
# (3/4/5) records which set that was.

# The match tiebreak (10pt/advantage-set decider) only ever occurs in the actual
# deciding set, so the match winner is always its winner.
df['won_match_tb'] = ((df['match_tb'] == 1) & (df['win'] == 1)).astype(int)

print('Matches with set-1 TB:           ', df.groupby('match_id')['has_tb_s1'].first().sum())
print('Matches with set-2 TB:           ', df.groupby('match_id')['has_tb_s2'].first().sum())
print('Matches with set-3 TB:           ', df.groupby('match_id')['has_tb_s3'].first().sum())
print('Matches with set-4 TB:           ', df.groupby('match_id')['has_tb_s4'].first().sum())
print('Matches with set-5 TB:           ', df.groupby('match_id')['has_tb_s5'].first().sum())
print('Matches with 10-pt/adv match TB: ', df.groupby('match_id')['match_tb'].first().sum())
print('  ...by deciding set:')
print(df.loc[df['match_tb'] == 1].groupby('match_id')['match_tb_set'].first().value_counts().sort_index())


Matches with set-1 TB:            462
Matches with set-2 TB:            473
Matches with set-3 TB:            210
Matches with set-4 TB:            31
Matches with set-5 TB:            6
Matches with 10-pt/adv match TB:  15
  ...by deciding set:
match_tb_set
3.0    8
5.0    7
Name: count, dtype: int64


## 3. Reshape to tiebreak-level

For each team-row, we create one tiebreak observation per tiebreak type present in that
match. The result is a long dataframe with columns `tb_set` (1-5) and `tb_type`
(7pt / 10pt) alongside all original team-level variables.


In [4]:
base_cols = [
    'match_id', 'tournament', 'year', 'surface', 'stage_code',
    'same_country', 'same_language', 'ling_prox',
    'rank_mean', 'opp_rank_mean', 'rank_gap', 'single_top100',
    'exp_mean', 'exp_mean_sq', 'win',
    'pre_olympic', 'olympic_period',
]

# Each entry: (filter_col, won_col, tb_set, tb_type). tb_set=None means "read the
# actual set number off match_tb_set per row" (needed only for the match_tb category,
# since that decider can be set 3, 4, or 5 depending on match length).
TB_DEFS = [
    ('has_tb_s1', 'won_tb_s1', 1, '7pt'),
    ('has_tb_s2', 'won_tb_s2', 2, '7pt'),
    ('has_tb_s3', 'won_tb_s3', 3, '7pt'),
    ('has_tb_s4', 'won_tb_s4', 4, '7pt'),
    ('has_tb_s5', 'won_tb_s5', 5, '7pt'),
    ('match_tb',  'won_match_tb', None, '10pt'),
]

chunks = []
for flag_col, won_col, tb_set, tb_type in TB_DEFS:
    sub = df[df[flag_col] == 1][base_cols + [won_col]].copy()
    sub = sub.rename(columns={won_col: 'won_tb'})
    if tb_set is None:
        sub['tb_set'] = df.loc[df[flag_col] == 1, 'match_tb_set'].astype(int).values
    else:
        sub['tb_set'] = tb_set
    sub['tb_type'] = tb_type
    chunks.append(sub)
    print(f'  {flag_col}: {len(sub)} team-obs ({len(sub)//2} tiebreaks)')

tb_panel = pd.concat(chunks, ignore_index=True)
tb_panel = tb_panel.sort_values(['match_id', 'tb_set', 'tb_type', 'win']).reset_index(drop=True)

print(f'\nTotal tiebreak team-obs: {len(tb_panel)}')
print(f'Unique tiebreaks (match x set x type): {len(tb_panel)//2}')


  has_tb_s1: 924 team-obs (462 tiebreaks)
  has_tb_s2: 946 team-obs (473 tiebreaks)
  has_tb_s3: 420 team-obs (210 tiebreaks)
  has_tb_s4: 62 team-obs (31 tiebreaks)
  has_tb_s5: 12 team-obs (6 tiebreaks)
  match_tb: 30 team-obs (15 tiebreaks)

Total tiebreak team-obs: 2394
Unique tiebreaks (match x set x type): 1197


## 4. Sanity checks

In [5]:
# Each (match_id, tb_set, tb_type) should have exactly 2 rows (one per team)
counts = tb_panel.groupby(["match_id", "tb_set", "tb_type"]).size()
incomplete = counts[counts != 2]
if len(incomplete) == 0:
    print("OK: every tiebreak has exactly 2 team rows")
else:
    print(f"WARNING: {len(incomplete)} tiebreak groups with != 2 rows")
    print(incomplete)

# won_tb should sum to 1 per tiebreak for complete pairs
won_sums = tb_panel.groupby(["match_id", "tb_set", "tb_type"])["won_tb"].sum()
assert (won_sums == 1).all(), f"won_tb sum != 1 for some tiebreaks: {won_sums[won_sums != 1]}"
print("OK: exactly one winner per tiebreak")

# Missing values in key regression variables
key_vars = ["won_tb", "same_country", "same_language", "ling_prox",
            "rank_mean", "opp_rank_mean", "exp_mean"]
missing_counts = tb_panel[key_vars].isna().sum()
print("")
print("Missing values in key variables:")
print(missing_counts[missing_counts > 0] if missing_counts.any() else "  None")

print("")
print("Tiebreak obs by type:")
print(tb_panel.groupby(["tb_set", "tb_type"]).size().rename("n_team_obs"))

OK: every tiebreak has exactly 2 team rows
OK: exactly one winner per tiebreak

Missing values in key variables:
  None

Tiebreak obs by type:
tb_set  tb_type
1       7pt        924
2       7pt        946
3       10pt        16
        7pt        420
4       7pt         62
5       10pt        14
        7pt         12
Name: n_team_obs, dtype: int64


## 5. Save

In [6]:
tb_panel.to_csv(OUT_PATH, index=False)
print(f'Saved {len(tb_panel)} rows → {OUT_PATH}')
print()
print('Column list:')
print(tb_panel.columns.tolist())
print()
print('Sample rows:')
tb_panel.head(6)

Saved 2394 rows → C:\Users\ALESSANDRO\Documents\GitHub\tennis-homophily\data\atp\tiebreak_panel.csv

Column list:
['match_id', 'tournament', 'year', 'surface', 'stage_code', 'same_country', 'same_language', 'ling_prox', 'rank_mean', 'opp_rank_mean', 'rank_gap', 'single_top100', 'exp_mean', 'exp_mean_sq', 'win', 'pre_olympic', 'olympic_period', 'won_tb', 'tb_set', 'tb_type']

Sample rows:


,match_id,tournament,year,surface,stage_code,same_country,same_language,ling_prox,rank_mean,opp_rank_mean,rank_gap,single_top100,exp_mean,exp_mean_sq,win,pre_olympic,olympic_period,won_tb,tb_set,tb_type
0,2,Australian Open,2018,Hard,2,1,1,1,87.5,55.5,13.0,0,5.0,25.00,0,0,0,1,1,7pt
1,2,Australian Open,2018,Hard,2,0,0,0,55.5,87.5,83.0,0,14.5,210.25,1,0,0,0,1,7pt
2,3,Australian Open,2018,Hard,2,0,0,0,472.5,35.0,749.0,0,11.0,121.00,0,0,0,0,1,7pt
3,3,Australian Open,2018,Hard,2,0,1,1,35.0,472.5,26.0,0,14.0,196.00,1,0,0,1,1,7pt
4,7,Australian Open,2018,Hard,2,0,1,1,20.0,79.5,10.0,0,12.5,156.25,0,0,0,0,1,7pt
5,7,Australian Open,2018,Hard,2,0,0,1,79.5,20.0,5.0,0,16.0,256.00,1,0,0,1,1,7pt
